## <u> Demonstrate some generic techniques for sonifying 1D data:</u>

**First, import relevant modules:**

In [ ]:
%reload_ext autoreload 
%autoreload 2
import matplotlib.pyplot as plt
from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss import channels
from strauss.score import Score
from strauss.generator import Synthesizer
import IPython.display as ipd
import os
from scipy.interpolate import interp1d
import numpy as np
%matplotlib inline

**Now, we construct some mock data!**

We use seeded random numbers to generate a mock 1D data set with features and noise:

In [ ]:
# seed the randoms...
np.random.seed(0)

# construct arrays of size N for x and y...
N = 300
x = np.linspace(0,1,N)
y = np.zeros(N)

# define a Gaussian function...
gauss = lambda x, m, s: np.exp(-(x-m)**2/s) 

# place some randomised gaussians...
for i in range(10):
    a,b,c = np.random.random(3)
    y += gauss(x, b, 1e-3*c) * a ** 3

# now add some noise and normalise
y += np.random.random(N) * y.mean()
y /= y.max()*1.2
y += 0.15


plt.plot(x,y)
plt.ylabel('Some dependent Variable')
plt.xlabel('Some independent Variable')

**Set up some universal sonification parameters and classes for the examples below**

For all examples we use the `Synthesizer` generator to create a 30 second, mono sonification.

In [ ]:
# specify audio system (e.g. mono, stereo, 5.1, ...)
system = "stereo"

# length of the sonification in s
length = 15.

# set up synth and turn on LP filter
generator = Synthesizer()
generator.load_preset('pitch_mapper')
generator.preset_details('pitch_mapper')

### <u>Example 1</u> &nbsp; **Pitch Mapping**

In [ ]:
notes = [["A2"]]
score =  Score(notes, length)

data = {'pitch':1.,
        'time_evo':x,
        'azimuth':(x*0.5+0.25) % 1,
        'polar':0.5,
        'pitch_shift':y**0.7}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions()

soni = Sonification(score, sources, generator, system)
soni.render()
soni.notebook_display()

### <u>Example 2</u> &nbsp; **Volume Mapping**

In [ ]:
notes = [["A2"]]
score =  Score(notes, length)

data = {'pitch':1.,
        'time_evo':x,
        'azimuth':(x*0.5+0.25) % 1,
        'polar':0.5,
        'volume':y**0.7}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions()

soni = Sonification(score, sources, generator, system)
soni.render()
soni.notebook_display()

### <u>Example 3</u> &nbsp; **Filter Cutoff Mapping**

In [ ]:
generator = Synthesizer()
generator.modify_preset({'filter':'on'})

In [ ]:
notes = [["C2","G2","C3","G3"]]
score =  Score(notes, length)

data = {'pitch':[0,1,2,3],
        'time_evo':[x]*4,
        'azimuth':[(x*0.5+0.25) % 1]*4,
        'polar':[0.5]*4,
        'cutoff':[y**0.8]*4}

# set up source
sources = Objects(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions()

soni = Sonification(score, sources, generator, system)
soni.render()
soni.notebook_display()

In [ ]:
generator = Synthesizer()
generator.load_preset('windy')

In [ ]:
data = {'pitch':[0,1,2,3],
        'time_evo':[x],
        'azimuth':[(x*0.5+0.25) % 1],
        'polar':[0.5],
        'cutoff':[y**0.8]}
sources = Objects(data.keys())
sources.fromdict(data)
sources.apply_mapping_functions()

soni = Sonification(score, sources, generator, system)
soni.render()
soni.notebook_display()

#### <u>Example 4</u> &nbsp; **Animate Mappings**

In [ ]:
# Plot mappings against time evolution
for key, values in sources.raw_mapping.items():
    if key == 'time_evo':
        continue
    if isinstance(values[0], np.ndarray) and len(values[0]) == len(sources.raw_mapping['time_evo'][0]):
        fig, ax1 = plt.subplots()
        ax1.set_xlabel('Time [s]') 
        ax2 = ax1.twinx() 
        ax1.plot(sources.raw_mapping['time_evo'][0], values[0])
        ax1.set_ylabel('Data')
        ax1.tick_params(axis ='y')
        ymin, ymax = ax1.get_ylim()
        ydel = (ymax-ymin)/(y.max() - y.min())
        yoff = ydel*(y.min()-ymin)/(ymax-ymin)
        ax2.set_ylim(0-yoff, ydel-yoff)
        ax2.set_ylabel(key)
        ax2.tick_params(axis ='y') 

In [ ]:
# Make frames for animations of mappings as a function of time. This may take several minutes.
import warnings
from pathlib import Path
from strauss.animation import Animate
import shutil

here = Path.cwd()

topdir = here / "figure_animations" / "1D" / "cutoff"
if topdir.exists():
    shutil.rmtree(topdir)
topdir.mkdir(parents=True, exist_ok=True)
if topdir.exists() and any(topdir.iterdir()):
    warnings.warn(f"{topdir} is not empty, instead name "
                  "an empty directory, or a new one.")
else:
    pipe = Animate(Path(here) / "figure_animations" / "1D" / "cutoff", pars={"background_video": str(Path(here) / "example_media" / "starfield.mov")})
    pipe.register(f'cutoff', sonification=soni, pre_caption=f'This video shows how cutoff is mapped to the data.', post_caption="Thank you for listening!", stype='animation')
    xp = sources.raw_mapping['time_evo'][0]
    yp = sources.raw_mapping['cutoff'][0]
    nframe = int(soni.score.length*int(pipe.pars['fps']))
    xf = np.linspace(xp[0], xp[-1], nframe)
    yf = np.interp(xf, xp, yp)
    xp, yp = xf, yf
    for i in range(xp.size)[::1]:
        fig, ax1 = plt.subplots()
        ax1.set_xlabel('Time [s]') 
        ax2 = ax1.twinx() 
        ax1.plot(xp, yp)
        ax1.set_ylabel('Data')
        ax1.tick_params(axis ='y')
        ax1.axvline(xp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ax1.axhline(yp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ymin, ymax = ax1.get_ylim()
        ydel = (ymax-ymin)/(y.max() - y.min())
        yoff = ydel*(y.min()-ymin)/(ymax-ymin)
        ax2.set_ylim(0-yoff, ydel-yoff)
        ax2.set_ylabel('Cutoff')
        ax2.tick_params(axis ='y')
        plt.savefig(pipe.frames['cutoff'].parent / f'frame_{i:05d}.png', dpi=120)
        plt.close()
    print(f"cutoff frames created!")
    pipe.render()

In [ ]:
from IPython.display import Video
Video(f"figure_animations/1D/cutoff/final.mp4", embed=True, width=960, height=540)